# 6.1 字元切詞有什麼成本？

# 第 6 章：自己的中英文 tokenizer

前置：字元ID與下一字loss。Tokenizer訓練是統計切詞規則，不是訓練LLM。早期與CLI使用可還原ByteTokenizer；BPE在本章獨立學習，換模型時要同步詞表與checkpoint。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：常見卡片合併：不是先懂詞義**

每次合併一對常見相鄰符號。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/bpe.svg")))

**先想一想：**同樣三個中文字與三個英文字母，UTF-8 bytes一樣多嗎？

一字一token很直覺，但英文常見詞可能拆成很多格；byte版本讓未知文字可還原，中文一字通常又需要多個byte。

**把直覺寫成數學：**序列長度影響attention的T×T矩陣；詞表大小影響embedding的V×D。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
texts = ["小小貓", "cat", "🙂"]
for s in texts:
    print(s, "字元", len(s), "UTF8 bytes", len(s.encode("utf-8")))

**如何讀結果：**字元、bytes、BPE token三個單位不能互換。

**只改一件事：**只加入一個emoji，看兩種長度差。
